## Chatbot with Profile Schema & LongTerm Memory

### Review:
- we built a chatbot with both chekpointer (withing thread) session based memory and store (across thread) long term mmeory agents previously
- agent saved semantic memory(facts and information or general knowlege ) about the user in the store
- we learnt to define namespace and use get and put methods to interact with the store.
- get - get memory
- put - write to the store
- memory was stored `in the hot path` as user was chatting or interactivng with it

### Goal: Chatbot storeing memory in a sturctured manner
- previously we stored the memory as a string seperated by comma
- In practice , memory must have structure which makes them easy to access and use
### Objective:
- Build a chatbot that saves semantic memory of the user in a **single user profile.**
- use **library and trustcall** to update the schema with new information

In [2]:
# Load API key
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_USE_V1"] = "true"

#Load lang graph tracing api
# set LangSmith tracing environment
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ['LANGCHAIN_PROJECT'] = 'langGraph-Course'

In [5]:
# create genai client and llm
from google import genai

client = genai.Client(api_key = os.environ["GOOGLE_API_KEY"])
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [7]:
# create a llm using any of the above models
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI( model= "gemini-2.5-flash" , 
                              temperature = 0.2 )
llm.invoke("What day is this?").content

'As an AI, I don\'t have a real-time clock or a concept of "today" in the same way humans do.\n\nTo find out the current day, you can check your device\'s clock or calendar!'

## Profile Schema
-Python has many different types for **structured data, such as TypedDict, Dictionaries, JSON, and Pydantic.**
- Use TypeDict for structured output

#### You are creating an instance of the UserProfile type.

✔ A type = the blueprint
UserProfile (your TypedDict) defines:
- what fields exist
- what types those fields must have
- It’s like a schema.

✔ An instance = a real object created from that blueprint
- user_profile is the actual data object that follows the schema.

TypedDict : “Here is what a user profile should look like.”

Instance : “Here is one actual user profile.”

TypedDict instances are used for:
- state objects
- memory stores
- tool inputs
- structured outputs
- agent context
They help ensure your data is predictable and type‑safe.

In [13]:
from typing_extensions import TypedDict , List

# create a class blue print schema for user profile
class Userprofile(TypedDict):
    user_name : str
    age:int
    interests:List[str] # list of user interests

# create an instance of **Userprofile** with real objects
user_profile : Userprofile = {
    "user_name" : 'Diya',
    "age":40,
    "interests": ["Loves reading", "gardening", "Swimming"]
}
user_profile

{'user_name': 'Diya',
 'age': 40,
 'interests': ['Loves reading', 'gardening', 'Swimming']}

## update the store schema
- Use method PUT
- 

In [14]:
# initialize in memory store
from langgraph.store.memory import InMemoryStore
store_memory = InMemoryStore()

# set up namespace for store
user_id ="1"
namespace = ("memory" , user_id)

# set key and value
key = "user_profile"
value = user_profile # schema instance

# update the store
store_memory.put(namespace , key , value)

### retrive memory:
1. use GET method
2. search the store to retrive values

In [15]:
# using get method
memory = store_memory.get(namespace , key)
memory

Item(namespace=['memory', '1'], key='user_profile', value={'user_name': 'Diya', 'age': 40, 'interests': ['Loves reading', 'gardening', 'Swimming']}, created_at='2026-06-03T03:52:27.093006+00:00', updated_at='2026-06-03T03:52:27.093006+00:00')

In [16]:
memory.dict()

{'namespace': ['memory', '1'],
 'key': 'user_profile',
 'value': {'user_name': 'Diya',
  'age': 40,
  'interests': ['Loves reading', 'gardening', 'Swimming']},
 'created_at': '2026-06-03T03:52:27.093006+00:00',
 'updated_at': '2026-06-03T03:52:27.093006+00:00'}

In [17]:
# search the store
search_memory = store_memory.search(namespace)
search_memory

[Item(namespace=['memory', '1'], key='user_profile', value={'user_name': 'Diya', 'age': 40, 'interests': ['Loves reading', 'gardening', 'Swimming']}, created_at='2026-06-03T03:52:27.093006+00:00', updated_at='2026-06-03T03:52:27.093006+00:00', score=None)]

In [19]:
search_memory[-1].dict()

{'namespace': ['memory', '1'],
 'key': 'user_profile',
 'value': {'user_name': 'Diya',
  'age': 40,
  'interests': ['Loves reading', 'gardening', 'Swimming']},
 'created_at': '2026-06-03T03:52:27.093006+00:00',
 'updated_at': '2026-06-03T03:52:27.093006+00:00',
 'score': None}